# Notebook 03 — CNN Classifier From Scratch (Stage 3)

Companion to `03-cnn-classification-architectures.md`. We procedurally generate a tiny,
fully synthetic image classification dataset — **"superscript-like"** crops (small digit,
shifted toward the top of the frame) vs. **"normal-text-like"** crops (larger digit, roughly
centered/lower) — and train the same lightweight CNN architecture sketched in Chapter 03 to
tell them apart. No real dataset, no GPU, and no pretrained weights are required.

> `pip install torch pillow numpy` — this uses **PyTorch** (CPU is entirely sufficient; the
> whole dataset is a few hundred 24x24 grayscale images and training is a handful of
> epochs, so this completes in a few seconds on CPU).


## 1. Generate the synthetic dataset

Each image is a 24x24 grayscale crop containing one digit glyph, drawn with PIL:

- **Class 1 ("superscript-like")**: small font, positioned near the *top* of the frame —
  standing in for a citation-marker-style crop (small + elevated, per Chapter 01/02).
- **Class 0 ("normal-text-like")**: larger font, positioned lower/more centered — standing
  in for a regular character or a false-positive candidate (e.g. a page number or exponent
  digit that YOLO proposed but isn't actually a citation).

This mirrors the true/false-positive distinction the CNN classifier is trained on in Chapter
03: the label is about geometry-driven visual appearance, not the digit's identity.


In [1]:
import random
import numpy as np
from PIL import Image, ImageDraw, ImageFont

IMG_SIZE = 24


def make_glyph_image(is_superscript, seed):
    """Renders one synthetic glyph crop. `is_superscript=True` -> small font, shifted toward
    the top of the frame (mimics a true citation-marker crop). `is_superscript=False` ->
    larger font, lower/more centered (mimics ordinary text or a false-positive candidate)."""
    rng = random.Random(seed)
    img = Image.new("L", (IMG_SIZE, IMG_SIZE), color=255)
    draw = ImageDraw.Draw(img)
    digit = str(rng.randint(0, 9))

    try:
        font_big = ImageFont.truetype("arial.ttf", 16)
        font_small = ImageFont.truetype("arial.ttf", 9)
    except OSError:
        font_big = ImageFont.load_default()
        font_small = ImageFont.load_default()

    if is_superscript:
        font = font_small
        x, y = rng.randint(2, 10), rng.randint(0, 4)
    else:
        font = font_big
        x, y = rng.randint(1, 6), rng.randint(4, 9)

    draw.text((x, y), digit, fill=0, font=font)

    arr = np.array(img, dtype=np.float32) / 255.0
    noise_level = rng.uniform(0.0, 0.03)
    arr = arr + np.random.RandomState(seed).normal(0, noise_level, arr.shape).astype(np.float32)
    return np.clip(arr, 0.0, 1.0)


def make_dataset(n_per_class, seed_offset):
    X, y = [], []
    for i in range(n_per_class):
        X.append(make_glyph_image(True, seed=seed_offset + i))
        y.append(1)  # 1 = superscript-like
        X.append(make_glyph_image(False, seed=seed_offset + 10_000 + i))
        y.append(0)  # 0 = normal-text-like
    X = np.stack(X)[:, None, :, :].astype(np.float32)  # add channel dim -> (N, 1, 24, 24)
    y = np.array(y, dtype=np.int64)
    return X, y


X_train, y_train = make_dataset(n_per_class=200, seed_offset=0)
X_test, y_test = make_dataset(n_per_class=50, seed_offset=50_000)

print("train:", X_train.shape, y_train.shape)
print("test: ", X_test.shape, y_test.shape)


train: (400, 1, 24, 24) (400,)
test:  (100, 1, 24, 24) (100,)


## 2. Define the tiny CNN

The same shape of network sketched in `03-cnn-classification-architectures.md`: a few small
conv+pool blocks followed by a linear classification head. It's deliberately small — the
input crops are tiny and the task is a simple binary decision, not general-purpose image
understanding, so a deep/heavy backbone would be pure overhead (and, per Chapter 03, overhead
that matters at inference time given how many YOLO candidates this classifier has to score
per page).


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)


class TinySuperscriptCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 24 -> 12
            nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 12 -> 6
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(16, 2)  # 2 classes: normal-text-like vs. superscript-like

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)


model = TinySuperscriptCNN()
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal trainable parameters: {n_params:,}")


TinySuperscriptCNN(
  (features): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): AdaptiveAvgPool2d(output_size=1)
  )
  (classifier): Linear(in_features=16, out_features=2, bias=True)
)

Total trainable parameters: 1,282


## 3. Train for a couple of epochs

Small dataset, tiny network, CPU-only Adam training — this finishes in a few seconds.


In [3]:
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

batch_size = 32
n_epochs = 8
n_train = X_train_t.shape[0]

for epoch in range(n_epochs):
    model.train()
    perm = torch.randperm(n_train)
    total_loss, correct = 0.0, 0

    for start in range(0, n_train, batch_size):
        idx = perm[start:start + batch_size]
        xb, yb = X_train_t[idx], y_train_t[idx]

        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(idx)
        correct += (logits.argmax(1) == yb).sum().item()

    train_acc = correct / n_train
    print(f"epoch {epoch + 1}/{n_epochs}  loss={total_loss / n_train:.4f}  train_acc={train_acc:.3f}")


epoch 1/8  loss=0.6911  train_acc=0.500
epoch 2/8  loss=0.6856  train_acc=0.677
epoch 3/8  loss=0.6803  train_acc=0.698
epoch 4/8  loss=0.6734  train_acc=0.820
epoch 5/8  loss=0.6687  train_acc=0.900


epoch 6/8  loss=0.6601  train_acc=0.940
epoch 7/8  loss=0.6522  train_acc=0.693
epoch 8/8  loss=0.6441  train_acc=0.917


## 4. Evaluate


In [4]:
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_preds = test_logits.argmax(1)
    test_acc = (test_preds == y_test_t).float().mean().item()

    true_pos = ((test_preds == 1) & (y_test_t == 1)).sum().item()
    false_pos = ((test_preds == 1) & (y_test_t == 0)).sum().item()
    false_neg = ((test_preds == 0) & (y_test_t == 1)).sum().item()
    precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else float("nan")
    recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) > 0 else float("nan")

print(f"Final test accuracy:  {test_acc:.3f}")
print(f"Precision (superscript-like): {precision:.3f}")
print(f"Recall    (superscript-like): {recall:.3f}")


Final test accuracy:  0.930
Precision (superscript-like): 0.877
Recall    (superscript-like): 1.000


## Takeaway

Even this tiny, from-scratch CNN, trained on a couple hundred synthetic examples for a few
seconds on CPU, should comfortably separate the two classes — because the underlying visual
signal (small + elevated vs. large + centered) is exactly the kind of local, low-level
pattern that a couple of conv+pool blocks are good at picking up (Chapter 03's receptive-field
discussion). In the real pipeline, this is the model that receives YOLO's (noisy,
over-inclusive) candidate crops and makes the final true-superscript-vs-false-positive call —
see Notebook 04 for how it plugs into the full sequential pipeline.
